# 26차시: [실습] 금융 뉴스 감성 분석 (Sentiment Analysis)

## 학습 목표
- 금융 뉴스 텍스트의 감성(긍정/부정) 분석
- 25차시에서 배운 임베딩 기법을 활용한 감성 분석 실습
- 뉴스 감성과 시장 심리 파악

## 학습 내용
1. 감성 분석이란?
2. 금융 뉴스 수집 (모듈 2 연계)
3. 임베딩 기반 감성 분석 (25차시 NLP 기법 활용)
4. 감성 점수 계산
5. 시각화 및 해석

In [5]:
!pip install -Uq koreanize-matplotlib langchain-huggingface

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from datetime import datetime
from IPython.display import display
import re
import requests
from bs4 import BeautifulSoup
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

---
## 1. 감성 분석이란?

### 정의
**감성 분석(Sentiment Analysis)**: 텍스트에서 긍정/부정/중립의 감정을 파악하는 기술

### 금융에서의 활용
| 활용 | 설명 |
|------|------|
| 시장 심리 파악 | 뉴스/SNS의 전반적 분위기 측정 |
| 투자 의사결정 | 부정적 뉴스 증가 시 리스크 관리 |
| 이벤트 분석 | 특정 사건에 대한 반응 분석 |

### 감성 분석 방법
1. **사전 기반**: 미리 정의된 긍정/부정 단어 사전 사용
2. **임베딩 기반**: 텍스트를 벡터로 변환하여 의미적 유사도 계산 (25차시)
3. **머신러닝 기반**: 레이블된 데이터로 모델 학습

---
## 2. 금융 뉴스 수집 (모듈 2 연계)

In [7]:
# 모듈 2에서 배운 크롤링 함수 (15차시)
# 네이버 금융(finance.naver.com)이 stock.naver.com 으로 개편되면서
# 화면이 자바스크립트로 그려지므로 HTML 파싱 대신 페이지가 호출하는 JSON API를 사용
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Referer': 'https://stock.naver.com/'
}

def crawl_financial_news(limit=20):
    """네이버 증권 주요 뉴스 수집"""
    url = "https://stock.naver.com/api/domestic/news/list"
    params = {'category': 'mainnews', 'page': 1, 'pageSize': limit}
    response = requests.get(url, params=params, headers=HEADERS, timeout=10)
    response.raise_for_status()
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    news_data = []
    for a in response.json().get('articles', [])[:limit]:
        summary = (a.get('subcontent') or '').strip()
        news_data.append({
            '제목': a['title'],
            '요약': summary[:100],
            '링크': f"https://n.news.naver.com/article/{a['officeId']}/{a['articleId']}",
            '수집시각': now
        })

    return pd.DataFrame(news_data)

In [8]:
# 뉴스 수집
print("[금융 뉴스 크롤링]")
print("=" * 60)

df_news = crawl_financial_news(limit=20)
print(f"수집된 뉴스: {len(df_news)}건")
display(df_news[['제목']].head(10))

[금융 뉴스 크롤링]
수집된 뉴스: 20건


,제목
0,코스피 4%나 뛰었는데…나 홀로 급락한 '이 종목' [종목+]
1,"“133% 올라야 겨우 본전”…160층에 갇힌 개미들, SK하닉 10%는 248층"
2,"힐튼호텔 재개발, 3.7조 본PF 완료했다…6개월 앞당긴 '속도전'"
3,"""수십만 원 배당 받으면 뭐해 수백이 날아갔는데"" 개미들은 안 즐겁다"
4,롤러코스터 증시에 ETF 괴리율 ‘폭증’…11월에만 1000건 돌파 [투자360]
5,"""내 노후자금 어쩌나"" 국민연금 적립금 202조 증발…1866조→1664조(종합)"
6,엔비디아 물타기 나선 서학개미…매수 늘어도 보관액 '뚝'
7,"""눈물 쏟고 고개 숙이고""…코스피 반토막에 전문가들도 '충격'"
8,‘트리플 할인’일 때 담아서 키워볼까…1년 만에 48% 오른 ETF [이번주 찜할 ...
9,삼성 휴머노이드 기대에 로봇株 질주…바이오도 불기둥


---
## 3. 임베딩 기반 감성 분석 (25차시 NLP 기법 활용)

### 임베딩을 활용한 감성 분석
25차시에서 배운 임베딩을 사용하여 텍스트의 의미를 벡터로 변환하고,
긍정/부정 기준 문장과의 유사도를 계산하여 감성을 판단합니다.

In [9]:
# 임베딩 모델 로드
embeddings = HuggingFaceEmbeddings(model_name="nlpai-lab/KURE-v1")

print("- 한국어 특화 임베딩 모델 (KURE-v1)")
print("- 텍스트를 의미 벡터로 변환하여 감성 분석")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\trimu\anaconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\trimu\.cache\huggingface\hub\models--nlpai-lab--KURE-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/807 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

- 한국어 특화 임베딩 모델 (KURE-v1)
- 텍스트를 의미 벡터로 변환하여 감성 분석


In [10]:
# 임베딩 기반 감성 분석
def analyze_sentiment_embedding(text, embeddings_model):
    """
    임베딩 기반: 의미적으로 유사한 문장과 비교

    Returns:
        dict: score, label
    """
    # 긍정/부정 기준 문장
    positive_ref = "주가 상승 실적 호조 전망 낙관"
    negative_ref = "주가 하락 실적 부진 전망 비관"

    # 텍스트와 기준 문장을 벡터로 변환
    text_vec = embeddings_model.embed_query(text)
    pos_vec = embeddings_model.embed_query(positive_ref)
    neg_vec = embeddings_model.embed_query(negative_ref)

    # 유사도 계산
    pos_sim = cosine_similarity([text_vec], [pos_vec])[0][0]
    neg_sim = cosine_similarity([text_vec], [neg_vec])[0][0]

    # 점수 계산 (-1 ~ 1)
    score = pos_sim - neg_sim

    # 레이블 결정
    if score > 0.1:
        label = '긍정'
    elif score < -0.1:
        label = '부정'
    else:
        label = '중립'

    return {
        'score': score,
        'label': label,
        'pos_similarity': pos_sim,
        'neg_similarity': neg_sim
    }

# 테스트
test_texts = [
    "삼성전자 3분기 영업이익 급등, 사상최고 실적 기대",
    "코스피 하락세 지속, 외국인 순매도 확대에 우려",
    "현대차 신차 출시, 시장 반응 주목"
]

print("[감성 분석 테스트]")
print("=" * 60)
for text in test_texts:
    result = analyze_sentiment_embedding(text, embeddings)
    print(f"\n텍스트: {text}")
    print(f"  → 점수: {result['score']:.2f}, 레이블: {result['label']}")
    print(f"  → 긍정 유사도: {result['pos_similarity']:.2f}, 부정 유사도: {result['neg_similarity']:.2f}")

[감성 분석 테스트]

텍스트: 삼성전자 3분기 영업이익 급등, 사상최고 실적 기대
  → 점수: 0.12, 레이블: 긍정
  → 긍정 유사도: 0.58, 부정 유사도: 0.46

텍스트: 코스피 하락세 지속, 외국인 순매도 확대에 우려
  → 점수: -0.19, 레이블: 부정
  → 긍정 유사도: 0.38, 부정 유사도: 0.57

텍스트: 현대차 신차 출시, 시장 반응 주목
  → 점수: 0.05, 레이블: 중립
  → 긍정 유사도: 0.46, 부정 유사도: 0.40


---
## 4. 뉴스 감성 점수 계산

In [11]:
# 전체 뉴스에 감성 분석 적용
def analyze_news_sentiment(df, embeddings_model):
    """뉴스 DataFrame에 감성 분석 결과 추가 (임베딩 기반)"""
    results = []

    for _, row in df.iterrows():
        # 제목 + 요약 합쳐서 분석
        text = row['제목'] + ' ' + str(row.get('요약', ''))

        # 임베딩 기반 감성 분석
        result = analyze_sentiment_embedding(text, embeddings_model)

        results.append(result)

    # 결과를 DataFrame에 추가
    df_result = df.copy()
    df_result['감성점수'] = [r['score'] for r in results]
    df_result['감성레이블'] = [r['label'] for r in results]
    df_result['긍정유사도'] = [r['pos_similarity'] for r in results]
    df_result['부정유사도'] = [r['neg_similarity'] for r in results]

    return df_result

# 감성 분석 적용
df_sentiment = analyze_news_sentiment(df_news, embeddings)

df_sentiment[['제목', '감성점수', '감성레이블', '긍정유사도', '부정유사도']]

,제목,감성점수,감성레이블,긍정유사도,부정유사도
0,코스피 4%나 뛰었는데…나 홀로 급락한 '이 종목' [종목+],-0.050972,중립,0.341894,0.392866
1,"“133% 올라야 겨우 본전”…160층에 갇힌 개미들, SK하닉 10%는 248층",-0.078834,중립,0.357998,0.436832
2,"힐튼호텔 재개발, 3.7조 본PF 완료했다…6개월 앞당긴 '속도전'",0.020024,중립,0.323011,0.302987
3,"""수십만 원 배당 받으면 뭐해 수백이 날아갔는데"" 개미들은 안 즐겁다",-0.071828,중립,0.345700,0.417527
4,롤러코스터 증시에 ETF 괴리율 ‘폭증’…11월에만 1000건 돌파 [투자360],-0.052072,중립,0.318639,0.370711
5,"""내 노후자금 어쩌나"" 국민연금 적립금 202조 증발…1866조→1664조(종합)",-0.072940,중립,0.307264,0.380204
6,엔비디아 물타기 나선 서학개미…매수 늘어도 보관액 '뚝',-0.010758,중립,0.384491,0.395248
7,"""눈물 쏟고 고개 숙이고""…코스피 반토막에 전문가들도 '충격'",-0.107733,부정,0.323111,0.430844
8,‘트리플 할인’일 때 담아서 키워볼까…1년 만에 48% 오른 ETF [이번주 찜할 ...,0.060487,중립,0.371759,0.311272
9,삼성 휴머노이드 기대에 로봇株 질주…바이오도 불기둥,0.123126,긍정,0.501273,0.378147


In [12]:
# 감성 분포 통계
print("[감성 분포 통계]")
print("=" * 60)

sentiment_counts = df_sentiment['감성레이블'].value_counts()
print(sentiment_counts)

avg_score = df_sentiment['감성점수'].mean()
print(f"\n평균 감성 점수: {avg_score:.3f}")

if avg_score > 0.05:
    market_sentiment = "긍정적 (낙관적)"
elif avg_score < -0.05:
    market_sentiment = "부정적 (비관적)"
else:
    market_sentiment = "중립적"

print(f"시장 심리: {market_sentiment}")

[감성 분포 통계]
감성레이블
중립    16
긍정     3
부정     1
Name: count, dtype: int64

평균 감성 점수: -0.007
시장 심리: 중립적


---
## 학습 정리

### 1. 감성 분석 파이프라인
```
뉴스 수집 → 텍스트 추출 → 임베딩 벡터 변환 → 유사도 계산 → 점수 계산 → 해석
```

### 2. 임베딩 기반 분석의 장단점
| 장점 | 한계 |
|------|------|
| 문맥 이해 가능 | 계산 비용 높음 |
| 의미적 유사도 측정 | 기준 문장 선택 중요 |
| 사전 구축 불필요 | 모델 로딩 시간 필요 |

---

### 다음 차시 예고
- 27차시: 생성형 AI / LLM을 활용한 투자 리서치
  - ChatGPT, Gemini 소개
  - LLM의 금융 분야 활용
  - 프롬프트 엔지니어링 기초